In [ ]:
import numpy as np
import pandas as pd

In [ ]:
nav=pd.read_csv("/content/02_nav_history.csv")

In [ ]:
print("Before cleaning:")
print(f"Shape: {nav.shape}")
print(nav.dtypes)
print(nav.head())

Before cleaning:
Shape: (46000, 3)
amfi_code      int64
date          object
nav          float64
dtype: object
   amfi_code        date      nav
0     119551  2022-01-03  54.3856
1     119551  2022-01-04  54.3474
2     119551  2022-01-05  54.6869
3     119551  2022-01-06  55.4550
4     119551  2022-01-07  55.3692


In [ ]:
nav['date'] = pd.to_datetime(nav['date'], errors='coerce')

In [ ]:
bad_dates = nav['date'].isna().sum()
print(f"Rows with unparseable dates: {bad_dates}")

Rows with unparseable dates: 0


In [ ]:
before = len(nav)
nav = nav.drop_duplicates(subset=['amfi_code', 'date'], keep='first')
after = len(nav)
print(f"Removed {before - after} duplicate rows")

Removed 0 duplicate rows


In [ ]:
nav = nav.sort_values(['amfi_code', 'date']).reset_index(drop=True)
nav.head(10)

,amfi_code,date,nav
0,100016,2022-01-03,520.4608
1,100016,2022-01-04,515.0971
2,100016,2022-01-05,521.7239
3,100016,2022-01-06,515.7880
4,100016,2022-01-07,515.1639
5,100016,2022-01-10,510.7136
6,100016,2022-01-11,513.5542
7,100016,2022-01-12,512.3195
8,100016,2022-01-13,510.2445
9,100016,2022-01-14,514.3636


In [ ]:
invalid_nav = nav[nav['nav'] <= 0]
print(f"Rows with NAV <= 0: {len(invalid_nav)}")
if len(invalid_nav) > 0:
    print(invalid_nav)

Rows with NAV <= 0: 0


In [ ]:
"""def fill_scheme_calendar(group):
    group = group.set_index('date')
    full_range = pd.date_range(start=group.index.min(), end=group.index.max(), freq='D')
    group = group.reindex(full_range)
    group['amfi_code'] = group['amfi_code'].ffill()
    group['nav'] = group['nav'].ffill()
    group.index.name = 'date'
    return group.reset_index()

nav_filled = nav.groupby('amfi_code', group_keys=False).apply(fill_scheme_calendar)"""

" def fill_scheme_calendar(group):\n    group = group.set_index('date')\n    full_range = pd.date_range(start=group.index.min(), end=group.index.max(), freq='D')\n    group = group.reindex(full_range)\n    group['amfi_code'] = group['amfi_code'].ffill()\n    group['nav'] = group['nav'].ffill()\n    group.index.name = 'date'\n    return group.reset_index()\n\nnav_filled = nav.groupby('amfi_code', group_keys=False).apply(fill_scheme_calendar)"

In [ ]:
def fill_scheme_calendar(group):
    code = group['amfi_code'].iloc[0]
    group = group.set_index('date')
    full_range = pd.date_range(start=group.index.min(), end=group.index.max(), freq='D')
    group = group.reindex(full_range)
    group['amfi_code'] = code
    group['nav'] = group['nav'].ffill()
    group.index.name = 'date'
    return group.reset_index()

nav_filled = (
    nav.groupby('amfi_code', group_keys=False)[['amfi_code', 'date', 'nav']]
    .apply(fill_scheme_calendar, include_groups=False)
)

print(f"Before fill: {len(nav)} rows")
print(f"After fill (calendar-complete): {len(nav_filled)} rows")

Before fill: 46000 rows
After fill (calendar-complete): 64320 rows


In [ ]:
still_invalid = nav_filled[nav_filled['nav'] <= 0]
print(f"Rows with NAV <= 0 after cleaning: {len(still_invalid)}")

remaining_nulls = nav_filled['nav'].isna().sum()
print(f"Remaining null NAV values: {remaining_nulls}")

Rows with NAV <= 0 after cleaning: 0
Remaining null NAV values: 0


In [ ]:
nav_filled['amfi_code'] = nav_filled['amfi_code'].astype(int)
print("After cleaning:")
print(f"Shape: {nav_filled.shape}")
print(nav_filled.dtypes)
print(nav_filled.head())

After cleaning:
Shape: (64320, 3)
date         datetime64[ns]
amfi_code             int64
nav                 float64
dtype: object
        date  amfi_code       nav
0 2022-01-03     100016  520.4608
1 2022-01-04     100016  515.0971
2 2022-01-05     100016  521.7239
3 2022-01-06     100016  515.7880
4 2022-01-07     100016  515.1639

Saved to data/processed/02_nav_history_cleaned.csv


In [ ]:
perf=pd.read_csv("/content/07_scheme_performance.csv")
print("Before cleaning:")
print(f"Shape: {perf.shape}")
print(perf.dtypes)
print(perf.head())
return_cols = ['return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct',
               'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct']

for col in return_cols:
    coerced = pd.to_numeric(perf[col], errors='coerce')
    bad_count = coerced.isna().sum() - perf[col].isna().sum()
    if bad_count > 0:
        print(f"[ISSUE] {col}: {bad_count} non-numeric value(s) found")
    else:
        print(f"[OK] {col}: all values are valid numeric")
print(perf[return_cols].describe())
high_sharpe = perf[perf['sharpe_ratio'] > 3]
print(high_sharpe[['amfi_code', 'scheme_name', 'category', 'sharpe_ratio', 'sortino_ratio']])
out_of_range = perf[(perf['expense_ratio_pct'] < 0.1) | (perf['expense_ratio_pct'] > 2.5)]
print(f"Schemes outside 0.1%–2.5%: {len(out_of_range)}")
print(out_of_range[['amfi_code', 'scheme_name', 'expense_ratio_pct']])

In [ ]:
import pandas as pd
txn = pd.read_csv("/content/08_investor_transactions.csv")
print("Before cleaning:")
print(f"Shape: {txn.shape}")
print(txn.dtypes)
print(txn.head())
txn["transaction_type"].value_counts()
print(txn['transaction_type'].value_counts(dropna=False))
invalid_amount = txn[txn['amount_inr'] <= 0]
print(f"Rows with amount_inr <= 0: {len(invalid_amount)}")
if len(invalid_amount) > 0:
    print(invalid_amount[['investor_id', 'transaction_type', 'amount_inr']].head(10))
print("Sample raw dates:", txn['transaction_date'].value_counts())
txn['transaction_date'] = pd.to_datetime(txn['transaction_date'], errors='coerce')
bad_dates = txn['transaction_date'].isna().sum()
print(f"Rows with unparseable dates: {bad_dates}")
print(f"Date range: {txn['transaction_date'].min()} to {txn['transaction_date'].max()}")
print(txn['kyc_status'].value_counts(dropna=False))

In [ ]:
fund_master = pd.read_csv('01_fund_master.csv')
print(f"Before: {fund_master.shape}")

fund_master['launch_date'] = pd.to_datetime(fund_master['launch_date'], errors='coerce')
fund_master = fund_master.drop_duplicates(subset=['amfi_code'])
print(f"Null launch dates: {fund_master['launch_date'].isna().sum()}")
print(f"Duplicate amfi_codes: {fund_master['amfi_code'].duplicated().sum()}")

fund_master.to_csv('01_fund_master_cleaned.csv', index=False)
print(f"After: {fund_master.shape} — saved")

In [ ]:
aum = pd.read_csv('03_aum_by_fund_house.csv')
print(f"Before: {aum.shape}")

aum['date'] = pd.to_datetime(aum['date'], errors='coerce')
aum = aum.drop_duplicates()
invalid_aum = aum[aum['aum_crore'] <= 0]
print(f"Rows with aum_crore <= 0: {len(invalid_aum)}")

aum.to_csv('03_aum_by_fund_house_cleaned.csv', index=False)
print(f"After: {aum.shape} — saved")

In [ ]:
sip = pd.read_csv('04_monthly_sip_inflows.csv')
print(f"Before: {sip.shape}")

sip = sip.drop_duplicates()
invalid_sip = sip[sip['sip_inflow_crore'] <= 0]
print(f"Rows with sip_inflow_crore <= 0: {len(invalid_sip)}")
print(f"Nulls in yoy_growth_pct (expected — first 12 months): {sip['yoy_growth_pct'].isna().sum()}")

sip.to_csv('04_monthly_sip_inflows_cleaned.csv', index=False)
print(f"After: {sip.shape} — saved")

In [ ]:
cat_inflows = pd.read_csv('05_category_inflows.csv')
print(f"Before: {cat_inflows.shape}")

cat_inflows = cat_inflows.drop_duplicates()
print(f"Nulls: {cat_inflows.isna().sum().sum()}")

cat_inflows.to_csv('05_category_inflows_cleaned.csv', index=False)
print(f"After: {cat_inflows.shape} — saved")

In [ ]:
folio = pd.read_csv('06_industry_folio_count.csv')
print(f"Before: {folio.shape}")

folio = folio.drop_duplicates()
print(f"Nulls: {folio.isna().sum().sum()}")

folio.to_csv('06_industry_folio_count_cleaned.csv', index=False)
print(f"After: {folio.shape} — saved")

In [ ]:
holdings = pd.read_csv('09_portfolio_holdings.csv')
print(f"Before: {holdings.shape}")

holdings['portfolio_date'] = pd.to_datetime(holdings['portfolio_date'], errors='coerce')
holdings = holdings.drop_duplicates()
invalid_weight = holdings[(holdings['weight_pct'] < 0) | (holdings['weight_pct'] > 100)]
print(f"Rows with invalid weight_pct: {len(invalid_weight)}")

holdings.to_csv('09_portfolio_holdings_cleaned.csv', index=False)
print(f"After: {holdings.shape} — saved")

In [ ]:
benchmark = pd.read_csv('10_benchmark_indices.csv')
print(f"Before: {benchmark.shape}")

benchmark['date'] = pd.to_datetime(benchmark['date'], errors='coerce')
benchmark = benchmark.drop_duplicates()
invalid_close = benchmark[benchmark['close_value'] <= 0]
print(f"Rows with close_value <= 0: {len(invalid_close)}")

benchmark.to_csv('10_benchmark_indices_cleaned.csv', index=False)
print(f"After: {benchmark.shape} — saved")